# Mini-TP 7 · Seguridad con SAIF — GraphQL

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por GraphQL**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** DoS por consultas profundas/costosas y **fuga del esquema por introspección**.
**Controles:** límite de profundidad/costo · introspección apagada · autorización por campo.
**Elementos SAIF:** automatizar defensas · bases sólidas.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.

### Dependencias
```bash
uv add strawberry-graphql fastapi httpx pyjwt pandas scikit-learn xgboost
```

> **Nota de versión.** El venv del curso tiene `strawberry-graphql 0.200`. En esa versión el
> código guiado original falla: `strawberry.Info` no existe (se importa de `strawberry.types`)
> y las extensiones pasadas como `lambda` dan `TypeError`. Por eso acá las extensiones se
> pasan como **instancias**, que funciona en esa versión.

## 0. Mi modelo
Igual que en los Mini-TP 1 a 6 (y en el Mini-TP 7 de REST), uso **mi propio modelo**: el XGBoost
**`xgb_best`** del TP integrador, que predice el **precio de autos usados** (dataset Car Dekho).
Lo entreno acá, reusando `MLOps1_final/common` sin copiarlo, para que el notebook corra **sin
servicios externos** (sin MLflow ni MinIO).

Además armo un **historial de predicciones** (los autos de test con su precio predicho). Es lo
que en el TP integrador expone la query `prediccionesStreaming`: una **lista** que el cliente
puede pedir por GraphQL, y que es justo lo que se usa para armar consultas costosas.

In [1]:
import os, sys, time, json, hashlib, datetime, logging
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Reusar el preprocesamiento del TP integrador (MLOps1_final/common)
TP_MLOPS1 = os.path.abspath(os.path.join("..", "TP_Final", "MLOps1_final"))
if TP_MLOPS1 not in sys.path:
    sys.path.insert(0, TP_MLOPS1)
DATASET_CSV = os.path.join(TP_MLOPS1, "dataset", "Car details v3.csv")

from common.constants import XGB_PARAMS, TRAIN_TEST_SIZE, TRAIN_TEST_SPLIT_SEED, MLFLOW_MODEL_NAME
from common.preprocessing import (
    extraer_columnas_tecnicas, drop_missing_technical_rows, drop_km_outliers,
    compute_top_brands, encode_categoricals, get_all_feature_columns,
    align_columns, build_feature_matrix, build_inference_features,
)
from common.schemas import CarRawInput

# Dataset y split estratificado por rango de precio (igual que en el TP integrador)
df = pd.read_csv(DATASET_CSV).drop_duplicates().reset_index(drop=True)
price_bins = pd.qcut(df["selling_price"], q=5,
                     labels=["muy_barato", "barato", "intermedio", "caro", "muy_caro"])
df_train, df_test = train_test_split(df, test_size=TRAIN_TEST_SIZE,
                                     stratify=price_bins, random_state=TRAIN_TEST_SPLIT_SEED)

# Feature engineering
df_train = drop_km_outliers(drop_missing_technical_rows(extraer_columnas_tecnicas(df_train)))
df_test = drop_missing_technical_rows(extraer_columnas_tecnicas(df_test))
top_brands = compute_top_brands(df_train)
df_train_enc, df_test_enc = encode_categoricals(df_train, df_test, top_brands)
all_features = get_all_feature_columns(df_train_enc)
df_train_enc, df_test_enc = align_columns(df_train_enc, df_test_enc, all_features)

X_train = build_feature_matrix(df_train_enc, all_features)
X_test = build_feature_matrix(df_test_enc, all_features)
y_train = np.log1p(df_train_enc["selling_price"])   # el modelo predice log(precio)

modelo = XGBRegressor(**XGB_PARAMS)
modelo.fit(X_train, y_train)

pred_test = np.expm1(modelo.predict(X_test))
METRICAS = {"r2": float(r2_score(df_test_enc["selling_price"], pred_test)),
            "mae": float(mean_absolute_error(df_test_enc["selling_price"], pred_test))}
MODEL_VERSION = f"{MLFLOW_MODEL_NAME}-local"
print(f"Modelo {MLFLOW_MODEL_NAME} listo. R2 test = {METRICAS['r2']:.4f} | MAE test = {METRICAS['mae']:,.0f}"
      f" | n_features = {len(all_features)}")

def features_de(car: CarRawInput) -> pd.DataFrame:
    return build_inference_features(car, top_brands, all_features)

def predecir_precio(car: CarRawInput) -> float:
    """Mismo camino que la API del TP integrador: auto crudo -> features -> precio."""
    return float(np.expm1(modelo.predict(features_de(car))[0]))

# Historial de predicciones: los 200 primeros autos de test con su precio predicho
COLS_AUTO = ["name", "year", "km_driven", "fuel", "seller_type", "transmission", "owner",
             "mileage", "engine", "max_power", "torque", "seats"]
HISTORIAL = [{"auto": a, "precio": float(p)}
             for a, p in zip(df_test[COLS_AUTO].head(200).to_dict("records"), pred_test[:200])]

# Auto de ejemplo (primer auto del dataset, precio real 450.000)
AUTO = {
    "name": "Maruti Swift Dzire VDI", "year": 2014, "km_driven": 145500,
    "fuel": "Diesel", "seller_type": "Individual", "transmission": "Manual",
    "owner": "First Owner", "mileage": "23.4 kmpl", "engine": "1248 CC",
    "max_power": "74 bhp", "torque": "190Nm@ 2000rpm", "seats": 5,
}
print(f"Precio predicho para el auto de ejemplo: {predecir_precio(CarRawInput(**AUTO)):,.0f}")
print(f"Historial de predicciones: {len(HISTORIAL)} autos")

# strawberry imprime cada error con su traceback completo; lo silencio para que la salida se lea
logging.getLogger("strawberry.execution").setLevel(logging.CRITICAL)

Modelo xgb_best listo. R2 test = 0.9372 | MAE test = 72,408 | n_features = 23
Precio predicho para el auto de ejemplo: 455,294
Historial de predicciones: 200 autos


## 1. Versión VULNERABLE — introspección abierta y sin límite de profundidad
El esquema es como el del TP integrador (`CarInput` con las features crudas del auto), más dos
cosas que un equipo agrega "porque es cómodo":
- `Modelo.hiperparametros` y `Prediccion.features` (el vector ya procesado que entra al XGBoost).
  Son **información sensible**: muestran cómo está hecho el modelo.
- Un **ciclo**: `Modelo.predicciones → Prediccion.modelo → Modelo ...`. En GraphQL es normal
  (es un grafo), pero permite armar consultas tan profundas como uno quiera.

Uso las clases de `common.schemas` para los enums, igual que el servicio GraphQL del TP integrador.

In [2]:
import strawberry
from typing import List, Optional
from strawberry.types import Info
from common.schemas import FuelEnum, SellerTypeEnum, TransmissionEnum, OwnerEnum

FuelType = strawberry.enum(FuelEnum, name="FuelType")
SellerType = strawberry.enum(SellerTypeEnum, name="SellerType")
TransmissionType = strawberry.enum(TransmissionEnum, name="TransmissionType")
OwnerType = strawberry.enum(OwnerEnum, name="OwnerType")

@strawberry.input
class CarInput:
    name: str
    year: int
    km_driven: int
    fuel: FuelType
    seller_type: SellerType
    transmission: TransmissionType
    owner: OwnerType
    mileage: str
    engine: str
    max_power: str
    torque: str
    seats: float

def a_car_raw(car: CarInput) -> CarRawInput:
    """CarInput de GraphQL -> CarRawInput de common (acá corre la validación de Pydantic)."""
    d = {k: getattr(car, k) for k in COLS_AUTO}
    return CarRawInput(**{k: (v.value if hasattr(v, "value") else v) for k, v in d.items()})

def gql_auto(a: dict) -> str:
    """Dict de auto -> literal GraphQL (los enums van sin comillas, con el nombre del miembro: diesel, first, ...)."""
    enums = {"fuel": FuelEnum, "seller_type": SellerTypeEnum, "transmission": TransmissionEnum, "owner": OwnerEnum}
    camel = lambda k: k.split("_")[0] + "".join(p.title() for p in k.split("_")[1:])
    partes = []
    for k, v in a.items():
        if k in enums:
            partes.append(f"{camel(k)}: {enums[k](v).name}")
        else:
            partes.append(f"{camel(k)}: {json.dumps(v)}")
    return "{" + ", ".join(partes) + "}"

# ---------------- esquema vulnerable ----------------
@strawberry.type
class MetricasV:
    r2: float
    mae: float

@strawberry.type
class PrediccionV:
    precio: float
    _auto: strawberry.Private[dict]

    @strawberry.field
    def features(self) -> List[float]:
        return features_de(CarRawInput(**self._auto)).iloc[0].astype(float).tolist()

    @strawberry.field
    def modelo(self) -> "ModeloV":
        return ModeloV()

@strawberry.type
class ModeloV:
    nombre: str = MLFLOW_MODEL_NAME
    version: str = MODEL_VERSION

    @strawberry.field
    def metricas(self) -> MetricasV:
        return MetricasV(**METRICAS)

    @strawberry.field
    def hiperparametros(self) -> str:
        return json.dumps(XGB_PARAMS)

    @strawberry.field
    def top_brands(self) -> List[str]:
        return top_brands

    @strawberry.field
    def predicciones(self) -> List[PrediccionV]:      # sin paginar: devuelve TODO
        return [PrediccionV(precio=h["precio"], _auto=h["auto"]) for h in HISTORIAL]

@strawberry.type
class QueryV:
    @strawberry.field
    def modelo(self) -> ModeloV:
        return ModeloV()

    @strawberry.field
    def predecir(self, car: CarInput) -> PrediccionV:
        raw = a_car_raw(car)
        return PrediccionV(precio=predecir_precio(raw), _auto=raw.model_dump(mode="json"))

schema_v = strawberry.Schema(QueryV)   # introspección ON, sin límite de profundidad

# a) Introspección: el atacante mapea TODA la API, incluidos los campos sensibles
r = schema_v.execute_sync("{ __schema { types { name fields { name } } } }")
tipos = {t["name"]: [f["name"] for f in (t["fields"] or [])]
         for t in r.data["__schema"]["types"] if not t["name"].startswith("__")}
print("Tipos expuestos por introspección:", len(r.data["__schema"]["types"]))
print("  ModeloV    ->", tipos["ModeloV"])
print("  PrediccionV->", tipos["PrediccionV"])

# b) Sin autenticación: cualquiera lee los hiperparámetros del modelo
r = schema_v.execute_sync("{ modelo { hiperparametros } }")
print("\nHiperparámetros, sin credenciales:", r.data["modelo"]["hiperparametros"][:80], "...")

# c) Consulta profunda: el ciclo modelo -> predicciones -> modelo multiplica el trabajo
for niveles, q in [(1, "{ modelo { predicciones { precio } } }"),
                   (2, "{ modelo { predicciones { modelo { predicciones { precio } } } } }")]:
    t0 = time.time(); r = schema_v.execute_sync(q); dur = time.time() - t0
    n = len(json.dumps(r.data))
    print(f"\n{niveles} nivel(es) de predicciones: {dur:6.2f} s, {n/1e6:6.2f} MB de respuesta")
print("  -> con 3 niveles serían 200^3 = 8 millones de objetos: una sola consulta tumba el servicio")

# d) Alias: 100 predicciones en UNA sola consulta (esquiva cualquier límite "por request")
alias = " ".join(f'p{km}: predecir(car: {gql_auto({**AUTO, "km_driven": km})}) {{ precio }}'
                 for km in range(0, 200_000, 2_000))
t0 = time.time(); r = schema_v.execute_sync("{ " + alias + " }")
print(f"\n100 predicciones con alias en 1 request: {len(r.data)} precios en {time.time()-t0:.2f} s"
      " (riesgo de extracción del modelo)")

Tipos expuestos por introspección: 21
  ModeloV    -> ['nombre', 'version', 'metricas', 'hiperparametros', 'topBrands', 'predicciones']
  PrediccionV-> ['precio', 'features', 'modelo']

Hiperparámetros, sin credenciales: {"n_estimators": 400, "max_depth": 6, "learning_rate": 0.029282661153638836, "su ...

1 nivel(es) de predicciones:   0.00 s,   0.01 MB de respuesta

2 nivel(es) de predicciones:   0.17 s,   1.01 MB de respuesta
  -> con 3 niveles serían 200^3 = 8 millones de objetos: una sola consulta tumba el servicio

100 predicciones con alias en 1 request: 100 precios en 1.39 s (riesgo de extracción del modelo)


## 2. Control principal (GUIADO) — auth por rol + sin introspección + límite de profundidad
El mismo control del ejemplo, aplicado a **mi** esquema: `predecir` recibe el auto real y exige
rol `analista` o `admin` (que viaja en el `context`), la introspección está apagada y la
profundidad máxima es 5.

In [3]:
from strawberry.extensions import QueryDepthLimiter, AddValidationRules
from graphql import NoSchemaIntrospectionCustomRule

@strawberry.type
class Query:
    @strawberry.field
    def modelo(self) -> ModeloV:
        return ModeloV()

    @strawberry.field
    def predecir(self, info: Info, car: CarInput) -> PrediccionV:
        # autorización por campo: el rol viaja en el context
        if info.context.get("rol") not in ("analista", "admin"):
            raise Exception("no autorizado")
        raw = a_car_raw(car)
        return PrediccionV(precio=predecir_precio(raw), _auto=raw.model_dump(mode="json"))

# En strawberry 0.200 las extensiones van como instancias (ver nota de versión arriba)
schema = strawberry.Schema(
    Query,
    extensions=[QueryDepthLimiter(max_depth=5),
                AddValidationRules([NoSchemaIntrospectionCustomRule])],
)

q_pred = f"{{ predecir(car: {gql_auto(AUTO)}) {{ precio }} }}"
sin_rol = schema.execute_sync(q_pred, context_value={})
con_rol = schema.execute_sync(q_pred, context_value={"rol": "analista"})
intro   = schema.execute_sync("{ __schema { types { name } } }", context_value={"rol": "analista"})
print("sin rol      :", sin_rol.errors[0].message)
print("con rol      :", con_rol.data)
print("introspección:", "bloqueada" if intro.errors else "ABIERTA")

profunda = "{ modelo { predicciones { modelo { predicciones { modelo { predicciones { precio } } } } } } }"
r = schema.execute_sync(profunda, context_value={"rol": "analista"})
print("profundidad 7:", r.errors[0].message if r.errors else "ACEPTADA")

# ¿Alcanza con la profundidad? No: estas dos consultas tienen profundidad <= 5 y pasan igual
t0 = time.time()
r = schema.execute_sync("{ modelo { predicciones { modelo { predicciones { precio } } } } }",
                        context_value={"rol": "analista"})
print(f"profundidad 5: {'rechazada' if r.errors else 'ACEPTADA'} -> 40.000 objetos en {time.time()-t0:.1f} s")
r = schema.execute_sync("{ " + alias + " }", context_value={"rol": "analista"})
print(f"100 alias    : {'rechazada' if r.errors else 'ACEPTADA'} -> {len(r.data)} predicciones en 1 request")
print("  sin rol, los hiperparámetros siguen a la vista:",
      schema.execute_sync("{ modelo { hiperparametros } }", context_value={}).data is not None)

sin rol      : no autorizado
con rol      : {'predecir': {'precio': 455294.0}}
introspección: bloqueada
profundidad 7: 'anonymous' exceeds maximum operation depth of 5
profundidad 5: ACEPTADA -> 40.000 objetos en 0.2 s
100 alias    : ACEPTADA -> 100 predicciones en 1 request
  sin rol, los hiperparámetros siguen a la vista: True


**Lo que el control guiado no cubre** (y resuelven los TODO):
- La **profundidad** no mide el **ancho**: una lista de 200 dentro de otra de 200 tiene
  profundidad 5 y son 40.000 objetos; 100 alias de `predecir` tienen profundidad 2. → **TODO 2**
- La autorización está **solo en `predecir`**: `hiperparametros` y `features` siguen abiertos. → **TODO 3**
- El rol viene en un `dict` que arma quien llama. En un servicio real tiene que salir de un
  **token verificado** (JWT), no de lo que diga el cliente. → lo hago al servir por HTTP.

## 3. Tu entrega — completa los `# TODO`

### TODO 1 — `predecir` con las features REALES de mi modelo
`predecir` recibe un `CarInput` (las 12 features crudas del auto, como en el TP integrador),
lo convierte a `CarRawInput` (validación de Pydantic: enums, año, asientos, km ≥ 0) y devuelve
la predicción del XGBoost entrenado. Si la validación falla, el cliente recibe un mensaje corto
y no el detalle interno.

### TODO 2 — límite de COSTO además del de profundidad
- Las listas **se paginan**: `predicciones(limit: Int = 10)`, con tope **50** en el resolver.
- Cada consulta se valida **antes de ejecutarse** con una regla que calcula su costo:
  cada campo cuesta 1, cada `predecir` cuesta **10** (corre el modelo), y lo que está dentro
  de una lista se multiplica por su `limit`. Si el costo pasa de **200**, se rechaza.
  Con eso caen las listas anidadas y los alias en masa, sin importar la profundidad.

### TODO 3 — autorización por campo más fina
Con `permission_classes` de strawberry:
- `predecir` → rol `analista` o `admin`.
- `hiperparametros`, `metricas` y `features` → solo `admin`. Si un analista los pide, recibe el
  resto de la respuesta y un error **solo en ese campo** (por eso son `Optional`).

### Gobernanza — logging de auditoría
Una extensión de strawberry registra **cada operación**: fecha y hora, usuario, rol, hash de la
consulta (no guardo el texto, puede traer datos del auto), costo calculado, cuántas
predicciones hizo y qué errores hubo (incluidos los rechazos).

### El rol sale de un JWT
Sirvo el esquema con FastAPI (`GraphQLRouter`, como en el TP integrador). El `context_getter`
lee el header `Authorization: Bearer ...`, verifica el JWT con `algorithms=["HS256"]` y pone en
el context el usuario y el rol **del token**. Sin token válido, el usuario queda anónimo.

In [4]:
import jwt   # uv add pyjwt
from graphql import ValidationRule, GraphQLError, FieldNode, FragmentSpreadNode, InlineFragmentNode, IntValueNode
from strawberry.permission import BasePermission
from strawberry.extensions import SchemaExtension

# ---------- TODO 2: costo de una consulta ----------
COSTO_MAX, LIMIT_DEFAULT, LIMIT_MAX = 200, 10, 50
LISTAS = {"predicciones"}           # campos que devuelven listas paginadas
COSTO_CAMPO = {"predecir": 10}      # campos caros (corren el modelo)

def costo_documento(document) -> int:
    fragmentos = {d.name.value: d for d in document.definitions if d.kind == "fragment_definition"}
    def costo_seleccion(sel_set, visitados=()):
        total = 0
        for sel in (sel_set.selections if sel_set else []):
            if isinstance(sel, FieldNode):
                propio = COSTO_CAMPO.get(sel.name.value, 1)
                hijos = costo_seleccion(sel.selection_set, visitados)
                if sel.name.value in LISTAS:
                    arg = next((a.value for a in sel.arguments or [] if a.name.value == "limit"), None)
                    # si el limit viene en una variable, asumo el peor caso
                    n = int(arg.value) if isinstance(arg, IntValueNode) else (LIMIT_DEFAULT if arg is None else LIMIT_MAX)
                    hijos *= min(max(n, 0), LIMIT_MAX)
                total += propio + hijos
            elif isinstance(sel, InlineFragmentNode):
                total += costo_seleccion(sel.selection_set, visitados)
            elif isinstance(sel, FragmentSpreadNode) and sel.name.value not in visitados:
                total += costo_seleccion(fragmentos[sel.name.value].selection_set, visitados + (sel.name.value,))
        return total
    return sum(costo_seleccion(d.selection_set) for d in document.definitions
               if d.kind == "operation_definition")

class LimiteCosto(ValidationRule):
    def enter_document(self, node, *_):
        costo = costo_documento(node)
        if costo > COSTO_MAX:
            self.report_error(GraphQLError(f"consulta demasiado costosa: {costo} > {COSTO_MAX}"))

# ---------- TODO 3: permisos por campo ----------
class EsAnalista(BasePermission):
    message = "no autorizado: requiere rol analista o admin"
    def has_permission(self, source, info, **kwargs):
        return info.context.get("rol") in ("analista", "admin")

class EsAdmin(BasePermission):
    message = "no autorizado: campo solo para admin"
    def has_permission(self, source, info, **kwargs):
        return info.context.get("rol") == "admin"

# ---------- Gobernanza: auditoría por operación ----------
AUDITORIA = []
class Auditoria(SchemaExtension):
    # on_operation envuelve TODA la operación: también registra las consultas que se
    # rechazan en la validación (introspección, costo, profundidad), que nunca llegan a ejecutarse
    def on_operation(self):
        yield
        ec = self.execution_context
        ctx = ec.context or {}
        errores = [e.message for e in (ec.errors or [])] + \
                  [e.message for e in ((ec.result.errors if ec.result else None) or [])]
        AUDITORIA.append({
            "ts": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "usuario": ctx.get("usuario"), "rol": ctx.get("rol"),
            "hash_consulta": hashlib.sha256(ec.query.encode()).hexdigest()[:16],
            "costo": costo_documento(ec.graphql_document) if ec.graphql_document else None,
            "predicciones": ctx.get("n_predicciones", 0),
            "errores": "; ".join(dict.fromkeys(errores))[:90] or None,
        })

# ---------- Esquema seguro ----------
@strawberry.type
class Metricas:
    r2: float
    mae: float

@strawberry.type
class Prediccion:
    precio: float
    _auto: strawberry.Private[dict]

    @strawberry.field(permission_classes=[EsAdmin])
    def features(self) -> Optional[List[float]]:
        return features_de(CarRawInput(**self._auto)).iloc[0].astype(float).tolist()

    @strawberry.field
    def modelo(self) -> "Modelo":
        return Modelo()

@strawberry.type
class Modelo:
    nombre: str = MLFLOW_MODEL_NAME
    version: str = MODEL_VERSION

    @strawberry.field(permission_classes=[EsAdmin])
    def metricas(self) -> Optional[Metricas]:
        return Metricas(**METRICAS)

    @strawberry.field(permission_classes=[EsAdmin])
    def hiperparametros(self) -> Optional[str]:
        return json.dumps(XGB_PARAMS)

    @strawberry.field
    def top_brands(self) -> List[str]:
        return top_brands

    @strawberry.field(permission_classes=[EsAnalista])
    def predicciones(self, limit: int = LIMIT_DEFAULT, offset: int = 0) -> Optional[List[Prediccion]]:
        limit = min(max(limit, 0), LIMIT_MAX)          # el tope también se aplica en el resolver
        return [Prediccion(precio=h["precio"], _auto=h["auto"]) for h in HISTORIAL[offset:offset + limit]]

@strawberry.type
class QuerySegura:
    @strawberry.field
    def modelo(self) -> Modelo:
        return Modelo()

    # TODO 1: predecir con las features reales del auto y el modelo entrenado
    @strawberry.field(permission_classes=[EsAnalista])
    def predecir(self, info: Info, car: CarInput) -> Optional[Prediccion]:
        try:
            raw = a_car_raw(car)
        except ValueError as e:
            raise Exception(f"entrada inválida: {e.errors()[0]['loc'][0]}: {e.errors()[0]['msg']}") from None
        info.context["n_predicciones"] = info.context.get("n_predicciones", 0) + 1
        return Prediccion(precio=round(predecir_precio(raw)), _auto=raw.model_dump(mode="json"))

schema_seg = strawberry.Schema(
    QuerySegura,
    extensions=[QueryDepthLimiter(max_depth=5),
                AddValidationRules([NoSchemaIntrospectionCustomRule, LimiteCosto]),
                Auditoria],
)
print("Esquema seguro listo")

Esquema seguro listo


In [5]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from strawberry.fastapi import GraphQLRouter

# En producción: secreto desde variable de entorno, nunca en el código
JWT_SECRET = os.environ.get("TP7_JWT_SECRET", "cambiar-en-produccion-con-32-bytes-o-mas")

def crear_token(usuario, rol, minutos=15, secreto=JWT_SECRET):
    exp = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=minutos)
    return jwt.encode({"sub": usuario, "rol": rol, "exp": exp}, secreto, algorithm="HS256")

def get_context(request: Request) -> dict:
    """El rol sale del JWT verificado, nunca de lo que mande el cliente en la consulta."""
    ctx = {"usuario": None, "rol": None}
    auth = request.headers.get("authorization", "")
    if auth.lower().startswith("bearer "):
        try:
            datos = jwt.decode(auth[7:], JWT_SECRET, algorithms=["HS256"], options={"require": ["exp", "sub"]})
            ctx.update(usuario=datos["sub"], rol=datos.get("rol"))
        except jwt.InvalidTokenError as e:
            ctx["usuario"] = f"token inválido ({type(e).__name__})"
    return ctx

app = FastAPI(title="Mini-TP 7 - GraphQL seguro")
app.include_router(GraphQLRouter(schema_seg, context_getter=get_context, graphiql=False), prefix="/graphql")
cli = TestClient(app)

def gql(query, token=None):
    h = {"Authorization": f"Bearer {token}"} if token else {}
    return cli.post("/graphql", json={"query": query}, headers=h).json()

def resumen(r):
    errs = [e["message"] for e in r.get("errors", [])]
    return ("ERROR: " + errs[0][:70]) if errs and not any(v for v in (r.get("data") or {}).values()) \
        else f"data={json.dumps(r.get('data'))[:70]}" + (f" | error parcial: {errs[0][:45]}" if errs else "")

t_ana, t_admin, t_inv = crear_token("jaime", "analista"), crear_token("ana", "admin"), crear_token("pepe", "invitado")
t_vencido = crear_token("jaime", "analista", minutos=-1)
t_falso = crear_token("hacker", "admin", secreto="otro-secreto-que-no-es-el-del-servidor!!")

q_pred = f"{{ predecir(car: {gql_auto(AUTO)}) {{ precio }} }}"
q_pred_feat = f"{{ predecir(car: {gql_auto(AUTO)}) {{ precio features }} }}"
q_basura = f"{{ predecir(car: {gql_auto({**AUTO, 'km_driven': -5, 'year': 3000})}) {{ precio }} }}"
q_hiper = "{ modelo { nombre version hiperparametros } }"
q_lista_ok = "{ modelo { predicciones(limit: 20) { precio } } }"
q_lista_anidada = "{ modelo { predicciones(limit: 50) { modelo { predicciones(limit: 50) { precio } } } } }"
q_lista_sin_limit = "{ modelo { predicciones { modelo { predicciones { precio } } } } }"
alias_15 = " ".join(f'p{i}: predecir(car: {gql_auto({**AUTO, "km_driven": 10_000*i})}) {{ precio }}' for i in range(15))
alias_25 = " ".join(f'p{i}: predecir(car: {gql_auto({**AUTO, "km_driven": 10_000*i})}) {{ precio }}' for i in range(25))

pruebas = [
    # (descripción, consulta, token)
    ("introspección (admin)",            "{ __schema { types { name } } }", t_admin),
    ("predecir sin token",               q_pred, None),
    ("predecir token vencido",           q_pred, t_vencido),
    ("predecir token con otro secreto",  q_pred, t_falso),
    ("predecir rol invitado",            q_pred, t_inv),
    ("predecir analista",                q_pred, t_ana),
    ("predecir analista + features",     q_pred_feat, t_ana),
    ("predecir admin + features",        q_pred_feat, t_admin),
    ("predecir datos inválidos",         q_basura, t_ana),
    ("hiperparámetros (analista)",       q_hiper, t_ana),
    ("hiperparámetros (admin)",          q_hiper, t_admin),
    ("lista paginada limit 20",          q_lista_ok, t_ana),
    ("listas anidadas limit 50x50",      q_lista_anidada, t_ana),
    ("listas anidadas sin limit",        q_lista_sin_limit, t_ana),
    ("15 alias de predecir",             "{ " + alias_15 + " }", t_ana),
    ("25 alias de predecir",             "{ " + alias_25 + " }", t_ana),
    ("profundidad 7",                    profunda, t_admin),
]
for nombre, q, tok in pruebas:
    print(f"  {nombre:34s} {resumen(gql(q, tok))}")

  introspección (admin)              ERROR: GraphQL introspection has been disabled, but the requested query conta
  predecir sin token                 ERROR: no autorizado: requiere rol analista o admin
  predecir token vencido             ERROR: no autorizado: requiere rol analista o admin
  predecir token con otro secreto    ERROR: no autorizado: requiere rol analista o admin
  predecir rol invitado              ERROR: no autorizado: requiere rol analista o admin
  predecir analista                  data={"predecir": {"precio": 455294.0}}
  predecir analista + features       data={"predecir": {"precio": 455294.0, "features": null}} | error parcial: no autorizado: campo solo para admin
  predecir admin + features          data={"predecir": {"precio": 455294.0, "features": [6.0, 145500.0, 23.4, 12
  predecir datos inválidos           ERROR: entrada inválida: year: Input should be less than or equal to 2026
  hiperparámetros (analista)         data={"modelo": {"nombre": "xgb_best", "ve

**Cómo leer los resultados:**
- La **introspección** está bloqueada incluso para `admin`. En desarrollo se puede prender con
  un esquema aparte, pero nunca en el servicio de producción.
- Sin token, con el token vencido o firmado con otro secreto, el usuario es **anónimo** y
  `predecir` se niega. El rol `invitado` tiene un token válido pero tampoco puede predecir.
- El **analista** predice, pero `features` y `hiperparametros` le vuelven en `null` con un error
  **solo en ese campo**. El **admin** los ve.
- Los **datos inválidos** (año 3000, km negativos) los frena `CarRawInput`, antes de llegar al modelo.
- **Costo** (máximo 200):
  - 20 precios paginados cuestan 22 y pasan.
  - Las listas anidadas **sin `limit`** usan el default de 10: son 10 × 10 = 100 objetos, cuestan
    122 y pasan. Es la misma consulta que en la versión vulnerable devolvía 40.000 objetos.
  - Con `limit: 50` en las dos listas cuestan 2.602 y se rechazan **antes de ejecutarse**.
  - Cada `predecir` cuesta 11 (10 del modelo + 1 de `precio`): 15 alias cuestan 165 y pasan;
    25 alias cuestan 275 y se rechazan.

In [6]:
# Registro de auditoría (en producción iría a un log append-only / data lake, no a una lista)
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
pd.DataFrame(AUDITORIA)

,ts,usuario,rol,hash_consulta,costo,predicciones,errores
0,2026-10-09T00:47:59+00:00,ana,admin,f3bcb57e2c511c60,3,0,"GraphQL introspection has been disabled, but the requested query contained the field '..."
1,2026-10-09T00:47:59+00:00,None,None,0bec0dd1fa1567d7,11,0,no autorizado: requiere rol analista o admin
2,2026-10-09T00:47:59+00:00,token inválido (ExpiredSignatureError),None,0bec0dd1fa1567d7,11,0,no autorizado: requiere rol analista o admin
3,2026-10-09T00:47:59+00:00,token inválido (InvalidSignatureError),None,0bec0dd1fa1567d7,11,0,no autorizado: requiere rol analista o admin
4,2026-10-09T00:47:59+00:00,pepe,invitado,0bec0dd1fa1567d7,11,0,no autorizado: requiere rol analista o admin
5,2026-10-09T00:47:59+00:00,jaime,analista,0bec0dd1fa1567d7,11,1,None
6,2026-10-09T00:47:59+00:00,jaime,analista,47786280a7d42b84,12,1,no autorizado: campo solo para admin
7,2026-10-09T00:47:59+00:00,ana,admin,47786280a7d42b84,12,1,None
8,2026-10-09T00:47:59+00:00,jaime,analista,69b82eae1159aee5,11,0,entrada inválida: year: Input should be less than or equal to 2026
9,2026-10-09T00:47:59+00:00,jaime,analista,cde6e71519ca6570,4,0,no autorizado: campo solo para admin


## 4. Reflexión

**1. ¿Por qué apagar la introspección en producción reduce el riesgo?**
Porque la introspección le entrega al atacante **el mapa completo de la API**: en la versión
vulnerable, una sola consulta me mostró todos los tipos y campos, incluidos `hiperparametros` y
`features`, que dicen cómo está construido el XGBoost. Sin el mapa, el atacante tiene que adivinar
nombres de campos, y cada intento fallido queda en la **auditoría**. No es un control suficiente
por sí solo (es "seguridad por oscuridad", y por eso además los campos sensibles piden rol
`admin`), pero reduce mucho la superficie de reconocimiento. Los clientes legítimos usan el
esquema publicado en la documentación, no la introspección.

**2. ¿Qué ataque frena el límite de profundidad/costo?**
- **DoS por consultas costosas:** con el ciclo `modelo → predicciones → modelo`, una consulta de
  pocas líneas pedía 40.000 objetos (y con un nivel más, 8 millones). El límite de **profundidad**
  corta los ciclos largos, pero no alcanza: el de **costo** mide también el **ancho** (listas y
  `limit`) y rechaza la consulta **antes de ejecutarla**, sin gastar CPU.
- **Extracción del modelo con alias:** en la versión vulnerable hice 100 predicciones en un solo
  request, variando los km. Eso esquiva cualquier rate limit "por request" y sirve para entrenar
  una copia del modelo. Como cada `predecir` cuesta 10, el costo limita cuántas predicciones
  entran en una consulta, y la auditoría registra cuántas hizo cada usuario.

**3. ¿Qué elemento de SAIF cubre mi control principal?**

| Control | Amenaza | Elemento de SAIF |
|---|---|---|
| Introspección apagada | reconocimiento de la API | **Bases sólidas de seguridad** |
| JWT verificado + permisos por campo (`EsAnalista`, `EsAdmin`) | acceso no autorizado, fuga de detalles del modelo | **Bases sólidas de seguridad** |
| Validación con `CarRawInput` | entradas fuera de dominio | **Bases sólidas de seguridad** |
| Límite de profundidad y de costo | DoS, extracción con alias | **Automatizar defensas** |
| Auditoría por operación (usuario, costo, predicciones, errores) | uso indebido sin rastro | **Detección y respuesta** + gobernanza |

El control principal (sin introspección + límite de profundidad/costo + autorización por campo)
cubre **bases sólidas** y **automatizar defensas**: son controles que se aplican solos en cada
consulta, en la etapa de validación, sin que nadie tenga que intervenir.